# WP1A — Benchmark reproduzível de métodos clássicos de ML no Tennessee Eastman Process
**Disciplina MEI0028/MEI0015 – Modelagem e Simulação · PUC Goiás · 2026 · Prof. Clarimar J. Coelho**

Base: Rieth et al. (2017), Harvard Dataverse, DOI 10.7910/DVN/6C3JR1 (domínio público) — 21 classes, 52 variáveis, 500 execuções/classe.
Protocolo: divisão **por execução (run)**; hiperparâmetros só na validação; teste lido uma única vez; 5 sementes; F1 macro + MCC; custo computacional medido no mesmo hardware.

Estes notebooks reproduzem exatamente o código executado. Cada célula `%%writefile` grava o script numerado; a célula seguinte o executa.

## Notebook 2 de 3 — Experimento do vazamento e seleção de hiperparâmetros
> Pré-requisito: notebook 1 executado (parquets + manifesto em `/content/ProjetoA_WP1A`).

In [ ]:
import os; os.makedirs("/content/ProjetoA_WP1A/src",exist_ok=True); os.chdir("/content/ProjetoA_WP1A")
os.environ["WP1A_ROOT"]="/content/ProjetoA_WP1A"; os.environ["NJOBS"]="2"
!pip -q install pyreadr pyarrow tabulate 2>/dev/null; print("ambiente pronto")

### 6. O experimento do vazamento (WP1A §18)
Mesmo modelo, mesmo volume de dados (25 runs/classe), duas condições: **(A)** divisão por execução — o protocolo; **(B)** divisão aleatória por amostra — a prática dominante na literatura, em que leituras vizinhas da mesma simulação caem em treino e teste. A diferença B − A mede o vazamento. Quatro variantes (árvore e floresta, com e sem poda) testam se o efeito cresce com a capacidade de memorização.

**Rótulo físico**: nos runs com falha, as 20 primeiras amostras (1 h) são fisicamente normais — recebem rótulo 0.

In [ ]:
%%writefile /content/ProjetoA_WP1A/src/04_piloto_vazamento.py
"""04_piloto_vazamento.py — Etapa 5 do plano: piloto com 25 runs/classe + experimento do vazamento.
Treina Árvore de Decisão duas vezes com o MESMO volume de dados:
  (A) divisão por run   — protocolo do WP1A
  (B) divisão aleatória por amostra — prática dominante na literatura (embaralha e sorteia linhas)
A diferença (B − A) é a medida direta do vazamento nos nossos dados."""
import os, json, time, numpy as np, pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, matthews_corrcoef
ROOT=os.environ.get("WP1A_ROOT") or os.path.join(os.path.dirname(os.path.abspath(__file__)),"..")
PROC=os.path.join(ROOT,"data","processed"); META=os.path.join(ROOT,"results","metadata"); TAB=os.path.join(ROOT,"results","tables")
SEED=42; N_RUNS=25
man=pd.read_csv(os.path.join(META,"manifesto_divisao.csv"))
tr_runs=man[man.conjunto=="treino"].groupby("faultNumber").head(N_RUNS)
va_runs=man[man.conjunto=="validacao"].groupby("faultNumber").head(N_RUNS)
def load(nome,runs):
    df=pd.read_parquet(os.path.join(PROC,nome+".parquet"))
    return df.merge(runs[["faultNumber","simulationRun"]],on=["faultNumber","simulationRun"])
tr=pd.concat([load("TEP_FaultFree_Training",tr_runs),load("TEP_Faulty_Training",tr_runs)])
va=pd.concat([load("TEP_FaultFree_Training",va_runs),load("TEP_Faulty_Training",va_runs)])
X_cols=[c for c in tr.columns if c not in ("faultNumber","simulationRun","sample")]
ONSET_TR=20
for d in (tr,va): d["y"]=np.where(d["sample"]<=ONSET_TR,0,d["faultNumber"]).astype(np.int32)
print(f"treino {len(tr):,} linhas | validação {len(va):,} linhas | {len(X_cols)} variáveis",flush=True)
def metr(y,p): return dict(acuracia=accuracy_score(y,p),acuracia_balanceada=balanced_accuracy_score(y,p),
                           f1_macro=f1_score(y,p,average="macro"),mcc=matthews_corrcoef(y,p))
res={}
MODELOS={"DT_leaf50":lambda: DecisionTreeClassifier(random_state=SEED,min_samples_leaf=50),
         "DT_leaf1_sem_poda":lambda: DecisionTreeClassifier(random_state=SEED,min_samples_leaf=1),
         "RF50_leaf50":lambda: RandomForestClassifier(n_estimators=50,min_samples_leaf=50,random_state=SEED,n_jobs=4),
         "RF50_leaf1_sem_poda":lambda: RandomForestClassifier(n_estimators=50,min_samples_leaf=1,random_state=SEED,n_jobs=4)}
sc=StandardScaler().fit(tr[X_cols]); Xtr_s=sc.transform(tr[X_cols]); Xva_s=sc.transform(va[X_cols])
todos=pd.concat([tr,va]); frac=len(tr)/len(todos)
Xa,Xb,ya,yb=train_test_split(todos[X_cols],todos.y,train_size=frac,random_state=SEED,shuffle=True,stratify=todos.y)
sc2=StandardScaler().fit(Xa); Xa_s=sc2.transform(Xa); Xb_s=sc2.transform(Xb)
for nome,mk in MODELOS.items():
    t=time.time(); m=mk().fit(Xtr_s,tr.y); A=metr(va.y,m.predict(Xva_s)); A["tempo_s"]=time.time()-t
    t=time.time(); m2=mk().fit(Xa_s,ya); B=metr(yb,m2.predict(Xb_s)); B["tempo_s"]=time.time()-t
    res[nome]=dict(A_por_run=A,B_por_amostra=B,diferenca_B_menos_A={k:B[k]-A[k] for k in ("acuracia","acuracia_balanceada","f1_macro","mcc")})
    print(nome,"por_run F1m=%.4f  por_amostra F1m=%.4f  Δ=%.4f"%(A["f1_macro"],B["f1_macro"],B["f1_macro"]-A["f1_macro"]),flush=True)
res["config"]=dict(runs_por_classe=N_RUNS,linhas_treino=int(len(tr)),linhas_validacao=int(len(va)),seed=SEED,rotulo="fisico (sample<=20 → normal)")
json.dump(res,open(os.path.join(META,"piloto_vazamento.json"),"w"),indent=2)
rows=[dict(modelo=n,condicao=c,**res[n][c]) for n in MODELOS for c in ("A_por_run","B_por_amostra")]+[dict(modelo=n,condicao="diferenca_B_menos_A",**res[n]["diferenca_B_menos_A"]) for n in MODELOS]
pd.DataFrame(rows).round(4).to_csv(os.path.join(TAB,"tab_piloto_vazamento.csv"),index=False)
print(json.dumps(res,indent=2)); print("PILOTO COMPLETO",flush=True)


In [ ]:
!python3 -u src/04_piloto_vazamento.py | tail -6 && cat results/tables/tab_piloto_vazamento.csv

### 7. SVM escalável: Nyström + gradiente estocástico em lotes
O SVM com núcleo RBF exato custa O(n²)–O(n³) e travou a máquina com 30 mil amostras. `LinearSVC` sobre a matriz densa de Nyström exige ~16 B/elemento (D = 1000 → > 8 GB). Esta classe aproxima o núcleo RBF (Nyström) e treina um SVM linear (perda *hinge*) por `partial_fit` em lotes de 100 mil linhas: memória ≈ lote × D (301 MB medidos).

In [ ]:
%%writefile /content/ProjetoA_WP1A/src/nystroem_sgd.py
"""nystroem_sgd.py — SVM linear escalável sobre aproximação de Nyström do núcleo RBF, treinado por gradiente
estocástico (perda hinge) em LOTES: a matriz de atributos transformados nunca é materializada por inteiro
(memória ≈ tamanho do lote × D). Substitui LinearSVC/liblinear, cuja representação esparsa de Φ densa exige
~16 bytes por elemento (D=1000 em 1,05 M linhas → > 8 GB). Implementações: scikit-learn (PEDREGOSA et al., 2011)."""
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import SGDClassifier
class NystroemSGD(BaseEstimator, ClassifierMixin):
    def __init__(self, n_components=500, alpha=1e-4, epochs=5, chunk=100_000, gamma=None, random_state=0, n_jobs=4):
        self.n_components=n_components; self.alpha=alpha; self.epochs=epochs; self.chunk=chunk; self.gamma=gamma; self.random_state=random_state; self.n_jobs=n_jobs
    def fit(self, X, y):
        X=np.asarray(X,dtype=np.float32); y=np.asarray(y); rng=np.random.RandomState(self.random_state)
        self.classes_=np.unique(y)
        self.nys_=Nystroem(kernel="rbf",gamma=self.gamma,n_components=self.n_components,random_state=self.random_state,n_jobs=self.n_jobs).fit(X)
        self.sgd_=SGDClassifier(loss="hinge",alpha=self.alpha,learning_rate="optimal",average=True,random_state=self.random_state,n_jobs=self.n_jobs,max_iter=1,tol=None)
        n=len(y)
        for ep in range(self.epochs):
            idx=rng.permutation(n)
            for s in range(0,n,self.chunk):
                b=idx[s:s+self.chunk]; Phi=self.nys_.transform(X[b]).astype(np.float32)
                self.sgd_.partial_fit(Phi,y[b],classes=self.classes_)
        return self
    def decision_function(self, X):
        X=np.asarray(X,dtype=np.float32); out=[]
        for s in range(0,len(X),self.chunk): out.append(self.sgd_.decision_function(self.nys_.transform(X[s:s+self.chunk]).astype(np.float32)))
        return np.vstack(out)
    def predict(self, X):
        X=np.asarray(X,dtype=np.float32); out=[]
        for s in range(0,len(X),self.chunk): out.append(self.sgd_.predict(self.nys_.transform(X[s:s+self.chunk]).astype(np.float32)))
        return np.concatenate(out)


### 8. Busca de hiperparâmetros — só na validação (WP1A §9.5)
Grade fatorial reduzida, 4 a 12 configurações por modelo (como Koçak et al., 2026). Critério: F1 macro na **validação**. O teste não é lido. `HP_MODELOS` permite dividir a busca entre máquinas: no projeto, o XGBoost rodou aqui (GPU T4) e os demais no Mac — a CPU do Colab é 5–10× mais lenta que um M3 para o scikit-learn, e **os tempos desta etapa não são usados** no artigo.

In [ ]:
%%writefile /content/ProjetoA_WP1A/src/06_busca_hp.py
"""06_busca_hp.py — Etapa 5 do WP1A (§9.5): seleção de hiperparâmetros SÓ na validação.
Treino: 50 runs/classe (subconjunto do 'treino' do manifesto) · Validação: 50 runs/classe.
Critério de seleção: F1 macro na validação. O TESTE NÃO É TOCADO.
Grade fatorial reduzida (4 a 12 configurações por modelo), avaliada só na validação.
Roda local ou no Colab (WP1A_ROOT aponta para a raiz do projeto)."""
import os, sys, json, time, itertools, numpy as np, pandas as pd, warnings; warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
import sys; sys.path.insert(0,os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.path.join(os.environ.get("WP1A_ROOT","."),"src"))
from nystroem_sgd import NystroemSGD
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, matthews_corrcoef, balanced_accuracy_score, accuracy_score
from xgboost import XGBClassifier
ROOT=os.environ.get("WP1A_ROOT") or os.path.join(os.path.dirname(os.path.abspath(__file__)),"..")
PROC=os.path.join(ROOT,"data","processed"); META=os.path.join(ROOT,"results","metadata"); CFG=os.path.join(ROOT,"configs")
SEED=42; N_TR=int(os.environ.get("HP_N_TREINO",50)); N_VA=50
SOMENTE=os.environ.get("HP_MODELOS")  # ex.: "xgboost,svm" para rodar só alguns
try:
    import subprocess; GPU=subprocess.run(["nvidia-smi","-L"],capture_output=True,text=True).returncode==0
except Exception: GPU=False
print(f"ROOT={ROOT} GPU={GPU} treino={N_TR} runs/classe",flush=True)
man=pd.read_csv(os.path.join(META,"manifesto_divisao.csv"))
tr_runs=man[man.conjunto=="treino"].groupby("faultNumber").head(N_TR); va_runs=man[man.conjunto=="validacao"].groupby("faultNumber").head(N_VA)
def load(nome,runs):
    return pd.read_parquet(os.path.join(PROC,nome+".parquet")).merge(runs[["faultNumber","simulationRun"]],on=["faultNumber","simulationRun"])
tr=pd.concat([load("TEP_FaultFree_Training",tr_runs),load("TEP_Faulty_Training",tr_runs)])
va=pd.concat([load("TEP_FaultFree_Training",va_runs),load("TEP_Faulty_Training",va_runs)])
X_cols=[c for c in tr.columns if c not in ("faultNumber","simulationRun","sample")]
ONSET_TR=20  # Rieth: falha introduzida 1 h (=20 amostras de 3 min) após o início nos runs de Training → sample<=20 é fisicamente normal
for d in (tr,va): d["y"]=np.where(d["sample"]<=ONSET_TR,0,d["faultNumber"]).astype(np.int32)
sc=StandardScaler().fit(tr[X_cols])  # ajustado SÓ no treino (§9.4)
Xtr=sc.transform(tr[X_cols]).astype(np.float32); ytr=tr.y.to_numpy(); Xva=sc.transform(va[X_cols]).astype(np.float32); yva=va.y.to_numpy()
print(f"treino {len(ytr):,} | validação {len(yva):,} | {len(X_cols)} vars",flush=True)
NJ=int(os.environ.get("NJOBS",4))
GRADES={
 "regressao_logistica": (lambda p: LogisticRegression(max_iter=300,n_jobs=NJ,**p),
    [dict(C=c) for c in (0.01,0.1,1.0,10.0)]),
 "arvore_decisao": (lambda p: DecisionTreeClassifier(random_state=SEED,**p),
    [dict(min_samples_leaf=l,max_depth=d) for l in (20,50,100,200) for d in (None,20)]),
 "random_forest": (lambda p: RandomForestClassifier(random_state=SEED,n_jobs=NJ,**p),
    [dict(n_estimators=n,min_samples_leaf=l,max_features=f) for n in (100,200) for l in (20,50,100) for f in ("sqrt",0.3)]),
 "gradient_boosting": (lambda p: HistGradientBoostingClassifier(random_state=SEED,early_stopping=False,**p),
    [dict(learning_rate=lr,max_iter=it,max_leaf_nodes=ln) for lr in (0.05,0.1) for it in (200,400) for ln in (31,63)]),
 "xgboost": (lambda p: XGBClassifier(random_state=SEED,tree_method="hist",device=("cuda" if GPU else "cpu"),n_jobs=NJ,subsample=0.8,colsample_bytree=0.8,**p),
    [dict(n_estimators=n,max_depth=d,learning_rate=lr) for n in (300,500) for d in (4,6) for lr in (0.05,0.1)]),
 "svm_nystroem": (lambda p: NystroemSGD(n_components=p["n_components"],alpha=p["alpha"],epochs=5,random_state=SEED,n_jobs=NJ),
    [dict(n_components=nc,alpha=a) for nc in (200,500,1000) for a in (1e-5,1e-4)]),
}
if SOMENTE: GRADES={k:v for k,v in GRADES.items() if k in SOMENTE.split(",")}
out_path=os.path.join(META,"busca_hp_resultados.json"); res=json.load(open(out_path)) if os.path.exists(out_path) else {}
for nome,(build,grade) in GRADES.items():
    res.setdefault(nome,[]); feitos={json.dumps(r["params"],sort_keys=True) for r in res[nome]}
    for params in grade:
        k=json.dumps(params,sort_keys=True)
        if k in feitos: continue
        t=time.time(); m=build(params); m.fit(Xtr,ytr); tt=time.time()-t; p=m.predict(Xva)
        r=dict(params=params,f1_macro=float(f1_score(yva,p,average="macro")),mcc=float(matthews_corrcoef(yva,p)),
               acuracia_balanceada=float(balanced_accuracy_score(yva,p)),acuracia=float(accuracy_score(yva,p)),tempo_treino_s=round(tt,1))
        res[nome].append(r); json.dump(res,open(out_path,"w"),indent=1)
        print(f"{nome:<22} {k:<70} F1m={r['f1_macro']:.4f} MCC={r['mcc']:.4f} {tt:6.0f}s",flush=True)
best={n:max(v,key=lambda r:r["f1_macro"]) for n,v in res.items() if v}
json.dump(best,open(os.path.join(CFG,"hiperparametros_escolhidos.json"),"w"),indent=2)
print("\nMELHORES (F1 macro na validação):"); [print(f"  {n:<22} {b['f1_macro']:.4f}  {b['params']}") for n,b in best.items()]
print("BUSCA HP COMPLETA",flush=True)


In [ ]:
import os; os.environ["HP_MODELOS"]="xgboost"; os.environ["HP_N_TREINO"]="50"
!python3 -u src/06_busca_hp.py

### Resultados obtidos (06/09/2026)
**Vazamento (Δ F1 macro, por amostra − por execução):** árvore com folha ≥ 50: +0,012 · árvore sem poda: +0,028 · RF folha ≥ 50: +0,013 · RF sem poda: +0,028. Efeito real, reprodutível e crescente com a capacidade — mas modesto com vetores instantâneos.

**Melhor F1 macro na validação por modelo:** regressão logística 0,4635 (C = 1) · árvore 0,7429 (folha ≥ 20, sem limite de profundidade; `max_depth=20` custou 8 pontos) · XGBoost 0,8135 (lr 0,1, profundidade 6, 500 árvores) · Random Forest, Gradient Boosting e SVM: ver `configs/hiperparametros_escolhidos.json`.